# Audio ML Foundations — Walkthrough
**Plan 1: ESC-50 CNN Classifier → Spectrogram VAE**

This notebook walks through the full pipeline:
1. Load a WAV file and plot the waveform
2. Convert to a log-mel spectrogram
3. Inspect the ESC50Dataset
4. Verify CNN model forward pass
5. Verify VAE model forward pass
6. (Post-training) Load checkpoints and explore

In [ ]:
# Add project root to path (run from notebooks/ directory)
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torchaudio
import matplotlib.pyplot as plt
import numpy as np

print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Load a WAV and plot the waveform

In [ ]:
# Change this path after running download_data.py
WAV_PATH = '../data/ESC-50-master/audio/1-100032-A-0.wav'

waveform, sr = torchaudio.load(WAV_PATH)
print(f'Sample rate: {sr}  |  Shape: {waveform.shape}  |  Duration: {waveform.shape[-1]/sr:.2f}s')

plt.figure(figsize=(12, 3))
plt.plot(waveform[0].numpy())
plt.title('Waveform')
plt.xlabel('Sample')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

## 2. Log-mel spectrogram

In [ ]:
import torchaudio.transforms as T
from src.dataset import SAMPLE_RATE, N_FFT, HOP_LENGTH, N_MELS

# Resample to 22050 Hz
if sr != SAMPLE_RATE:
    waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)

mel_transform = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
)
mel     = mel_transform(waveform)      # (1, 128, T)
log_mel = torch.log1p(mel)

print(f'Spectrogram shape: {log_mel.shape}')

plt.figure(figsize=(12, 4))
plt.imshow(log_mel[0].numpy(), aspect='auto', origin='lower', cmap='magma')
plt.colorbar(format='%+2.0f')
plt.title('Log-Mel Spectrogram')
plt.xlabel('Time frames')
plt.ylabel('Mel bins')
plt.tight_layout()
plt.show()

## 3. ESC50Dataset

In [ ]:
from src.dataset import ESC50Dataset, build_loaders

ds = ESC50Dataset(root='../data', fold=1, augment=False)
print(f'Fold 1 size: {len(ds)} samples')

spec, label = ds[0]
print(f'Spectrogram shape: {spec.shape}  |  Label: {label}')

## 4. CNN Classifier — forward pass check

In [ ]:
from src.cnn_classifier import CNNClassifier

model = CNNClassifier(num_classes=50)
dummy = torch.randn(4, 1, 128, 216)
out   = model(dummy)
print(f'CNN output shape: {out.shape}')   # expect (4, 50)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {params:,}')

## 5. Spectrogram VAE — forward pass check

In [ ]:
from src.vae import SpectrogramVAE, vae_loss

vae   = SpectrogramVAE(latent_dim=128)
dummy = torch.randn(4, 1, 128, 128)
x_hat, mu, logvar = vae(dummy)
print(f'x_hat: {x_hat.shape}  mu: {mu.shape}')

total, recon, kl = vae_loss(x_hat, dummy, mu, logvar, beta=1e-3)
print(f'Loss: total={total:.4f}  recon={recon:.4f}  kl={kl:.4f}')
params = sum(p.numel() for p in vae.parameters() if p.requires_grad)
print(f'Parameters: {params:,}')

## 6. Post-training: Load CNN checkpoint and evaluate

In [ ]:
# Run after training: python -m src.train_cnn --fold 1 --epochs 50

import os
CKPT = '../checkpoints/cnn_fold1_best.pt'

if os.path.exists(CKPT):
    ckpt  = torch.load(CKPT, map_location='cpu')
    model = CNNClassifier()
    model.load_state_dict(ckpt['model'])
    model.eval()
    print(f"Loaded checkpoint. Best val acc: {ckpt['metric']:.4f}")
else:
    print('No checkpoint found yet — train the CNN first.')

## 7. Post-training: Sample from the VAE

In [ ]:
# Run after training: python -m src.train_vae --epochs 100

from src.utils import spec_to_wav
import torchaudio

VAE_CKPT = '../checkpoints/vae_best.pt'

if os.path.exists(VAE_CKPT):
    ckpt = torch.load(VAE_CKPT, map_location='cpu')
    vae  = SpectrogramVAE(latent_dim=ckpt.get('latent_dim', 128))
    vae.load_state_dict(ckpt['model'])
    vae.eval()

    with torch.no_grad():
        samples = vae.sample(4, device='cpu')   # (4, 1, 128, 128)

    fig, axes = plt.subplots(1, 4, figsize=(16, 3))
    for i, ax in enumerate(axes):
        ax.imshow(samples[i, 0].numpy(), aspect='auto', origin='lower', cmap='magma')
        ax.set_title(f'Sample {i+1}')
        ax.axis('off')
    plt.suptitle('VAE Random Samples (latent space)')
    plt.tight_layout()
    plt.show()
else:
    print('No VAE checkpoint found yet — train the VAE first.')